In [1]:
# SYNTHETIC SCENE SETUP
import ivt
from ivt.scene import Scene, DiffuseBRDF, MicrofacetBRDF, Mesh, \
    PerspectiveCamera, HDRFilm, Integrator, EnvironmentLight
from ivt.io import read_image, read_mesh, write_image
from ivt.sampling import sample_sphere

from matplotlib import pyplot as plt
import torch
import numpy as np

scene = Scene()

# material
diffuse = read_image("assets/wood_olive/wood_olive_wood_olive_diffuse.png").dot([0.2989, 0.5870, 0.1140])
specular = read_image("assets/wood_olive/wood_olive_wood_olive_specular.png").dot([0.2989, 0.5870, 0.1140])
diffuse = np.stack([diffuse]*3, axis=-1)
specular = np.stack([specular]*3, axis=-1)
roughness = read_image("assets/wood_olive/wood_olive_wood_olive_roughness.png")
print(diffuse.shape)

scene.set('mat', MicrofacetBRDF(diffuse, specular, roughness))
scene.set('mesh1', Mesh.from_file("assets/meshes/sphere.obj", 'mat')) # surface to reconstruct
scene.set('film', HDRFilm(256, 256))
scene.set('integrator', Integrator('collocated', {'intensity':1}))
# scene.set('cam1', PerspectiveCamera.from_lookat(fov=40, origin=(1,0.5,0), target=(0,0,0), up=(0,1,0))) # camera

# randomly generate cameras
num_sensors = 50
fov = 40
radius = 1
target = (0,0,0)
up = (0,1,0)
for i, origin in enumerate(sample_sphere(num_sensors, radius, 'fibonacci')):
    scene.set(f'sensor {i}', PerspectiveCamera.from_lookat(fov, origin, target, up))

# render target images
renderer = ivt.renderer.Renderer(connector_name='psdr_jit', render_options = {
    "spp": 16,
    "sppe": 0,
    "sppse": 0,
    "npass": 1,
    "log_level": 0
})
# because ivt has a glitch that doesn't update changes in spp, workaround here to render a higher spp target image
I_t = torch.stack([renderer(scene, sensor_ids=range(num_sensors)) for _ in range(8)]).mean(axis=0)

# Visualize a target Image
def showimg(I):
    plt.imshow((I**0.454).detach().cpu().numpy().squeeze(), vmax=1)
showimg(I_t[10])

(1024, 1024, 3)


KeyError: 'psdr_jit'
  In call to configurable 'Renderer' (<class 'ivt.renderer.Renderer'>)

In [2]:
# likelihood of images
def logpdf(imgs):
    from math import sqrt, pi
    sigma = torch.tensor(0.02)
    # log of density of a normal distribution
    lpdf = ((imgs - I_t).pow(2.0)/(2.0*sigma.pow(2.0)) + \
                (sqrt(2*pi)*sigma).log()).sum()
    return lpdf

def U(xo):
    scene['mat']['d'] = xo[0]
    scene['mat']['s'] = xo[1]
    scene['mat']['r'] = xo[2]
    img = renderer(scene)
    lpdf = logpdf(img)
    return lpdf, img

In [3]:
logpdf(I_t)

tensor(-29423214., device='cuda:0')

In [16]:
I1 = renderer(scene, sensor_ids=range(num_sensors))

In [17]:
logpdf(I1)

tensor(-29271782., device='cuda:0')

In [ ]:
from tqdm import tqdm
import time

# compute posterior probability on a regular 3D grid
res = 24
Us = []
for i in tqdm(range(res)):
    for j in range(res):
        for k in range(res):
            x = (i+0.5)/res
            y = (j+0.5)/res
            z = (k+0.5)/res
            u = U_orig(torch.tensor([x,y,z]).cuda())[0].cpu()
            u = (-u).exp()
            Us.append(u)

In [18]:
max([2,3,4])

4